In [4]:
# [Célula 1: Imports e Leitura]
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
import seaborn as sns
import os

In [5]:
# Configurações visuais
sns.set_theme(style="whitegrid", palette="dark")

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")  # Carrega variáveis de ambiente do .env
DIR_DATAPROCESSED = ROOT / os.getenv("DIR_DATAPROCESSED")

df = pd.read_parquet(os.path.join(DIR_DATAPROCESSED, "preprocessado.parquet"))
df = pd.DataFrame(df)

meses = sorted(df['periodo'].unique())
df_atual = df[df['periodo'] == meses[-1]]
df_ant = df[df['periodo'] == meses[-2]]


In [6]:

# [Célula 2: Cálculo da Matriz de Agressividade]
# Cruzando Volume (Tamanho) com Crescimento (Agressividade)
vol_atual = df_atual.groupby('Empresa')['acessos'].sum()
vol_ant = df_ant.groupby('Empresa')['acessos'].sum()

# Preenchendo NaNs com 0 para evitar divisão por erro
cresc_pct = ((vol_atual - vol_ant) / vol_ant * 100).fillna(0)

matriz = pd.DataFrame({'Volume': vol_atual, 'Crescimento %': cresc_pct}).reset_index()
matriz_filtrada = matriz[matriz['Volume'] > 500] # Corte técnico

# Estatísticas descritivas para justificar os quadrantes do gráfico
print("--- Estatísticas da Matriz de Agressividade ---")
print(matriz_filtrada.describe())

mediana_crescimento = matriz_filtrada['Crescimento %'].median()
print(f"\nMediana de Crescimento do Mercado: {mediana_crescimento:.2f}%")
print("Empresas acima da mediana são consideradas de comportamento agressivo neste período.")

# [Célula 3: Análise de Market Share Tecnológico]
# Como as tecnologias estão distribuídas entre os top players?
top5_empresas = vol_atual.nlargest(5).index.tolist()

df_tech = df_atual[df_atual['Empresa'].isin(top5_empresas)]
tech_pivot = df_tech.pivot_table(index='Empresa', columns='Tecnologia Geração', values='acessos', aggfunc='sum', fill_value=0)

# Convertendo para percentual da própria empresa (Share of Wallet tecnológico)
tech_share = tech_pivot.div(tech_pivot.sum(axis=1), axis=0) * 100
print("\n--- Mix Tecnológico dos Top 5 Players (%) ---")
print(tech_share.round(1))

--- Estatísticas da Matriz de Agressividade ---
             Volume  Crescimento %
count  1.300000e+01      13.000000
mean   2.448222e+06       7.027204
std    2.938098e+06      13.051264
min    1.769100e+04      -2.631289
25%    1.870260e+05       0.953338
50%    1.264263e+06       3.488341
75%    3.607309e+06       6.476803
max    8.639275e+06      48.123418

Mediana de Crescimento do Mercado: 3.49%
Empresas acima da mediana são consideradas de comportamento agressivo neste período.

--- Mix Tecnológico dos Top 5 Players (%) ---
Tecnologia Geração       2G    3G     4G   5G
Empresa                                      
ALGAR (CTBC TELECOM)    0.0   0.0  100.0  0.0
CLARO                  22.0  38.2   39.3  0.5
DATORA                100.0   0.0    0.0  0.0
TIM                     1.8   0.7   97.5  0.0
VIVO                   53.4   9.6   36.8  0.2
